# Self-Referencing Table or Adjacency List Model

In [51]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("surya") \
    .config(
        "spark.jars.packages",
        "graphframes:graphframes:0.8.2-spark3.3-s_2.12"
    )\
    .getOrCreate()


In [52]:
data = [
    (1, "CEO", None),
    (2, "VP Eng", 1),
    (3, "VP Sales", 1),
    (4, "Eng Manager", 2),
    (5, "Engineer", 4)
]

df = spark.createDataFrame(data, ["emp_id", "emp_name", "manager_id"])

df.show()
df.createOrReplaceTempView("employees")

+------+-----------+----------+
|emp_id|   emp_name|manager_id|
+------+-----------+----------+
|     1|        CEO|      NULL|
|     2|     VP Eng|         1|
|     3|   VP Sales|         1|
|     4|Eng Manager|         2|
|     5|   Engineer|         4|
+------+-----------+----------+



In [53]:
# self Join 

from pyspark.sql.functions import col

df.alias("e") \
  .join(
      df.alias("m"),
      col("e.manager_id") == col("m.emp_id"),
      "left"
  ) \
  .select(
      col("e.emp_name").alias("employee"),
      col("m.emp_name").alias("manager")
  ) \
  .show()


+-----------+-----------+
|   employee|    manager|
+-----------+-----------+
|        CEO|       NULL|
|     VP Eng|        CEO|
|   VP Sales|        CEO|
|Eng Manager|     VP Eng|
|   Engineer|Eng Manager|
+-----------+-----------+



## Getting Full Hierarchy (Recursive Query) 

In [54]:
spark.sql("""
WITH RECURSIVE org AS (
    SELECT emp_id, emp_name, manager_id, 0 AS level
    FROM employees
    WHERE manager_id IS NULL
    
    UNION ALL
    
    SELECT e.emp_id, e.emp_name, e.manager_id, o.level + 1
    FROM employees e
    JOIN org o
      ON e.manager_id = o.emp_id
)
SELECT * FROM org;""").show()
          


+------+-----------+----------+-----+
|emp_id|   emp_name|manager_id|level|
+------+-----------+----------+-----+
|     1|        CEO|      NULL|    0|
|     2|     VP Eng|         1|    1|
|     3|   VP Sales|         1|    1|
|     4|Eng Manager|         2|    2|
|     5|   Engineer|         4|    3|
+------+-----------+----------+-----+




## Without recursive

### Spark 2 DataFrame Solution

In [55]:
from pyspark.sql.functions import lit

In [56]:
hierarchy = (
    df
    .filter(col("manager_id").isNull())
    .withColumn("level", lit(0))
)


In [57]:
current = hierarchy
result = hierarchy

max_depth = 10  # safety limit

for i in range(1, max_depth):
    next_level = (
        df.alias("e")
        .join(
            current.alias("p"),
            col("e.manager_id") == col("p.emp_id"),
            "inner"
        )
        .select(
            col("e.emp_id"),
            col("e.emp_name"),
            col("e.manager_id"),
            lit(i).alias("level")
        )
    )

    # Stop if no more children
    if next_level.count() == 0:
        break

    result = result.union(next_level)
    current = next_level


In [58]:
result.orderBy("level", "emp_id").show()


+------+-----------+----------+-----+
|emp_id|   emp_name|manager_id|level|
+------+-----------+----------+-----+
|     1|        CEO|      NULL|    0|
|     2|     VP Eng|         1|    1|
|     3|   VP Sales|         1|    1|
|     4|Eng Manager|         2|    2|
|     5|   Engineer|         4|    3|
+------+-----------+----------+-----+



### Add Hierarchy Path (Very Useful)

In [59]:
from pyspark.sql.functions import concat_ws

hierarchy = (
    df
    .filter(col("manager_id").isNull())
    .withColumn("level", lit(0))
    .withColumn("path", col("emp_name"))
)

current = hierarchy
result = hierarchy
max_depth = 5
for i in range(1, max_depth):
    next_level = (
        df.alias("e")
        .join(
            current.alias("p"),
            col("e.manager_id") == col("p.emp_id")
        )
        .select(
            col("e.emp_id"),
            col("e.emp_name"),
            col("e.manager_id"),
            lit(i).alias("level"),
            concat_ws(" → ", col("p.path"), col("e.emp_name")).alias("path")
        )
    )
    if next_level.count() == 0:
        break
    result = result.union(next_level)
    current = next_level


In [60]:
result.show()

+------+-----------+----------+-----+--------------------+
|emp_id|   emp_name|manager_id|level|                path|
+------+-----------+----------+-----+--------------------+
|     1|        CEO|      NULL|    0|                 CEO|
|     2|     VP Eng|         1|    1|        CEO → VP Eng|
|     3|   VP Sales|         1|    1|      CEO → VP Sales|
|     4|Eng Manager|         2|    2|CEO → VP Eng → En...|
|     5|   Engineer|         4|    3|CEO → VP Eng → En...|
+------+-----------+----------+-----+--------------------+

